# Planetary Rover GRPO Training (Colab)

This notebook runs the same GRPO pipeline as `train.py` for the OpenEnv hackathon submission.

Pipeline:
- Llama 3.2 1B + Unsloth 4-bit NF4 + LoRA
- TRL `GRPOTrainer`
- Two-tier rewards: format gatekeeper + environment reward from FastAPI physics server

## 1) Prepare Runtime



In [21]:
import os
import pathlib
import subprocess

# Define the target directory
repo_path = pathlib.Path('/content/planetary-rover-navigation')

# Clone the repo if it doesn't exist
if not repo_path.exists():
    print("Cloning repository...")
    subprocess.run(["git", "clone", "https://github.com/Gurram-Bhaskar/planetary-rover-navigation.git", str(repo_path)], check=True)
else:
    print("Repository already exists.")

# Change into the directory
os.chdir(repo_path)
print('Working directory:', os.getcwd())

Repository already exists.
Working directory: /content/planetary-rover-navigation


In [22]:
# 1. Install Unsloth using its official Colab installer (handles most heavy lifting)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# 2. Install core libraries WITHOUT forcing an old version of xformers.
# This allows pip to grab the pre-compiled version compatible with Colab's Torch.
!pip install --no-deps xformers trl peft accelerate bitsandbytes

# 3. Dynamically change strict "==" to flexible ">=" in your requirements.txt
!sed -i 's/==/>=/g' requirements.txt

# 4. Install the remaining app dependencies quietly
!pip install -q -r requirements.txt

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-vct0obon/unsloth_3a90bc8a4323461992d28a8f394e78dd
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-vct0obon/unsloth_3a90bc8a4323461992d28a8f394e78dd
  Resolved https://github.com/unslothai/unsloth.git to commit efed5c37394a144349cd9b1ea525e132e04584e5
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [23]:
import requests
import subprocess
import sys
import time

server_proc = subprocess.Popen([
    sys.executable, '-m', 'uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '7860'
])

deadline = time.time() + 30
server_ready = False
while time.time() < deadline:
    try:
        res = requests.get('http://127.0.0.1:7860/tasks', timeout=2)
        if res.ok:
            server_ready = True
            break
    except Exception:
        pass
    time.sleep(1)

if not server_ready:
    raise RuntimeError('Server failed to start within 30 seconds.')

print('FastAPI environment server is live on :7860')

FastAPI environment server is live on :7860


## 2) Run GRPO Training


In [24]:
import os
import wandb
from google.colab import userdata

try:
    # Try to grab the W&B key from Colab Secrets (the key icon on the left)
    wandb_key = userdata.get('WANDB_API_KEY')
    wandb.login(key=wandb_key)
    print("Successfully logged into Weights & Biases!")
except userdata.SecretNotFoundError:
    print("No WANDB_API_KEY found in Colab secrets. Disabling W&B logging for this run.")
    os.environ['WANDB_MODE'] = 'disabled'


No WANDB_API_KEY found in Colab secrets. Disabling W&B logging for this run.


In [ ]:
import os
import train as rover_train

os.environ['ROVER_SERVER_URL'] = 'http://127.0.0.1:7860'
os.environ['ROVER_NUM_GENERATIONS'] = '8'

rover_train.SERVER_URL = os.environ['ROVER_SERVER_URL']
rover_train.NUM_GENERATIONS = int(os.environ['ROVER_NUM_GENERATIONS'])

rover_train.main()

==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-1b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 450 | Num Epochs = 2 | Total steps = 900
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformer

Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / format_reward_fn / mean,rewards / format_reward_fn / std,rewards / environment_reward_fn / mean,rewards / environment_reward_fn / std
1,0.000001,0.000000,0.000000,30.625000,9.000000,54.000000,0.000000,30.625000,9.000000,54.000000,0.000019,0.000000,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.000000,44.750000,31.000000,95.000000,0.000000,44.750000,31.000000,95.000000,0.000008,0.000000,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.000000,38.000000,30.000000,54.000000,0.000000,38.000000,30.000000,54.000000,0.000009,0.000000,0.000000,0.000000,0.000000
4,0.000000,0.000000,0.000000,60.500000,28.000000,127.000000,0.000000,60.500000,28.000000,127.000000,0.000007,0.000000,0.000000,0.000000,0.000000
5,0.000000,0.000000,0.000000,46.250000,28.000000,71.000000,0.000000,46.250000,28.000000,71.000000,0.000011,0.000000,0.000000,0.000000,0.000000
6,0.000001,0.000000,0.000000,34.250000,4.000000,68.000000,0.000000,34.250000,4.000000,68.000000,0.000016,0.000000,0.000000,0.000000,0.000000
7,0.000000,0.000000,0.000000,47.375000,23.000000,123.000000,0.000000,47.375000,23.000000,123.000000,0.000011,0.000000,0.000000,0.000000,0.000000
8,0.000000,0.127488,0.360590,53.375000,25.000000,131.000000,0.000000,53.375000,25.000000,131.000000,0.000007,0.125000,0.353553,0.002488,0.007037
9,0.000000,0.374175,0.516879,38.125000,31.000000,43.000000,0.000000,38.125000,31.000000,43.000000,0.000012,0.375000,0.517549,-0.000825,0.022049
10,0.000000,0.000000,0.000000,33.250000,28.000000,39.000000,0.000000,33.250000,28.000000,39.000000,0.000010,0.000000,0.000000,0.000000,0.000000


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

In [ ]:
if 'server_proc' in globals() and server_proc.poll() is None:
    server_proc.terminate()
    print('Stopped FastAPI environment server.')
else:
    print('No running server process found.')